# 01 — Ingestion & Embedding

Loads the 8 Zepto policy documents, chunks them, embeds each chunk locally with
`sentence-transformers/all-MiniLM-L6-v2`, and stores the vectors in a persistent
ChromaDB collection (`zepto_policies`).

No API key and no LLM call are used anywhere in this notebook — embeddings run
entirely on-device.

This notebook also writes out `ingest.py`, a plain module that `graph.py` imports
so the same ingestion/retrieval logic is reusable outside the notebook (and inside
the Docker image, which cannot execute `.ipynb` files directly).


In [4]:

%%writefile ingest.py
"""
Ingestion + embedding module for the Zepto policy-assistant RAG pipeline.

Stage: ingestion -> embedding  (see README architecture section)

- load_documents(): reads docs/doc_*.txt off disk
- chunk_document(): one chunk per document (each doc is short and topically
  self-contained, so per-document chunking is the simplest correct scheme —
  explicitly allowed by the assignment spec)
- build_or_load_collection(): embeds each chunk with all-MiniLM-L6-v2 and
  upserts it into a persistent ChromaDB collection on disk (./chroma_db)
- retrieve_top_k(): embeds a query and returns the top-k most similar chunks
  by cosine similarity — this always runs for real (no MOCK_LLM branch here),
  since embeddings/ChromaDB need no API key and no network call at query time.
"""
import os
import glob
from typing import List, Dict

import chromadb
from sentence_transformers import SentenceTransformer

DOCS_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "docs")
CHROMA_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "chroma_db")
COLLECTION_NAME = "zepto_policies"
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"

_model = None


def get_embedder() -> SentenceTransformer:
    """Lazily load the local embedding model (downloaded once, cached by HF)."""
    global _model
    if _model is None:
        _model = SentenceTransformer(EMBED_MODEL_NAME)
    return _model


def load_documents(docs_dir: str = DOCS_DIR) -> Dict[str, str]:
    """Read all docs/doc_*.txt files. Returns {doc_id: text}."""
    paths = sorted(glob.glob(os.path.join(docs_dir, "doc_*.txt")))
    if not paths:
        raise FileNotFoundError(
            f"No doc_*.txt files found in {docs_dir}. "
            "Did you run the corpus-creation cell in 01_ingest.ipynb?"
        )
    docs = {}
    for path in paths:
        doc_id = os.path.splitext(os.path.basename(path))[0]  # e.g. 'doc_01'
        with open(path, "r", encoding="utf-8") as f:
            docs[doc_id] = f.read().strip()
    return docs


def chunk_document(doc_id: str, text: str, max_chars: int = 400) -> List[Dict]:
    """
    Chunk a single document. Each source doc here is a single short policy
    paragraph (a few sentences), so we default to one chunk per document.
    If a document exceeds max_chars, fall back to a fixed-size split so the
    scheme still degrades gracefully on longer corpora.
    """
    if len(text) <= max_chars:
        return [{"chunk_id": f"{doc_id}_c0", "doc_id": doc_id, "text": text}]

    chunks = []
    for i, start in enumerate(range(0, len(text), max_chars)):
        piece = text[start:start + max_chars].strip()
        if piece:
            chunks.append({"chunk_id": f"{doc_id}_c{i}", "doc_id": doc_id, "text": piece})
    return chunks


def build_or_load_collection(persist_dir: str = CHROMA_DIR):
    """
    Embed every chunk from every document and upsert into a persistent
    ChromaDB collection. Safe to re-run: upsert is idempotent on chunk_id.
    """
    client = chromadb.PersistentClient(path=persist_dir)
    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )

    docs = load_documents()
    all_chunks = []
    for doc_id, text in docs.items():
        all_chunks.extend(chunk_document(doc_id, text))

    embedder = get_embedder()
    texts = [c["text"] for c in all_chunks]
    embeddings = embedder.encode(texts, normalize_embeddings=True).tolist()

    collection.upsert(
        ids=[c["chunk_id"] for c in all_chunks],
        embeddings=embeddings,
        documents=texts,
        metadatas=[{"doc_id": c["doc_id"]} for c in all_chunks],
    )
    return collection


def get_collection(persist_dir: str = CHROMA_DIR):
    """Get the collection, building it first if it doesn't exist yet."""
    client = chromadb.PersistentClient(path=persist_dir)
    try:
        collection = client.get_collection(COLLECTION_NAME)
        if collection.count() == 0:
            return build_or_load_collection(persist_dir)
        return collection
    except Exception:
        return build_or_load_collection(persist_dir)


def retrieve_top_k(query: str, k: int = 3, persist_dir: str = CHROMA_DIR) -> List[Dict]:
    """
    Embed `query` and return the top-k most similar chunks via cosine
    similarity search in ChromaDB. Always real (no mock branch) — see
    module docstring.
    """
    collection = get_collection(persist_dir)
    embedder = get_embedder()
    query_embedding = embedder.encode([query], normalize_embeddings=True).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k,
    )

    hits = []
    ids = results.get("ids", [[]])[0]
    docs_ = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    dists = results.get("distances", [[]])[0] if results.get("distances") else [None] * len(ids)

    for chunk_id, text, meta, dist in zip(ids, docs_, metas, dists):
        hits.append({
            "chunk_id": chunk_id,
            "doc_id": meta.get("doc_id"),
            "text": text,
            "distance": dist,
        })
    return hits


if __name__ == "__main__":
    build_or_load_collection()
    print("Collection built. Example query:")
    for h in retrieve_top_k("How long do I have to return a damaged item?", k=3):
        print(f"  [{h['chunk_id']}] dist={h['distance']:.4f}  {h['text'][:80]}...")


Writing ingest.py


In [10]:
!pip3 install chromadb
!pip3 install sentence-transformer


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement sentence-transformer (from versions: none)
ERROR: No matching distribution found for sentence-transformer

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# Build the ChromaDB collection now (embeds all 8 docs locally, no API key/network needed
# beyond the one-time model download of all-MiniLM-L6-v2 from Hugging Face on first run).
from ingest import build_or_load_collection, retrieve_top_k

collection = build_or_load_collection()
print(f"Collection '{collection.name}' now has {collection.count()} chunks.")


c:\Users\Srivatsav\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Srivatsav\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9089.28it/s

Collection 'zepto_policies' now has 15 chunks.


In [13]:
# Sanity check: retrieval should return chunks from the correct source document.
for h in retrieve_top_k("How long do I have to report a spoiled item?", k=3):
    print(h["chunk_id"], h["doc_id"], round(h["distance"], 4), "->", h["text"][:90], "...")


doc_06_c0 doc_06 0.4006 -> If an order arrives with damaged, spoiled, or missing items, customers must report it with ...
doc_02_c0 doc_02 0.5146 -> Grocery and perishable items may be reported for a return within 24 hours of delivery if d ...
doc_06_c1 doc_06 0.6004 -> must be submitted through the report form before a replacement or refund is processed. ...


Expected: the top hit for a "spoiled item" query should come from `doc_06`
(Damaged or Missing Items) or `doc_02` (Returns & Refunds), since those are the
only documents that mention "spoiled" and the 24-hour reporting window.